# **Hamilton Filter - Creation de l'output GAP**

In [9]:
from DATALAKE.data import *
import statsmodels.api as sm
import matplotlib.pyplot as plt
import pandas as pd 
import numpy as np
import plotly.graph_objects as go

data_detrended = import_parquet('main_detrended')

In [16]:
# CODE 

data_detrended["log_gdp"] = np.log(data_detrended["gdp_nominal"])

df_hamilton = data_detrended

# Params Hamilton

h = 2     # filtrage sur window de 2 ans
p = 4     # Lag de 4 périodes 

# Construction du vect de lag
lags_cols = []
for i in range(p):
    col_name = f"lag_{h+i}"
    df_hamilton[col_name] = df_hamilton["log_gdp"].shift(h + i)
    lags_cols.append(col_name)

# Anticipation du PIB future par la regression 
df_hamilton["y_future"] = df_hamilton["log_gdp"].shift(-h)

# Regression hailton
reg_data = df_hamilton[["log_gdp"] + lags_cols].dropna()

X = sm.add_constant(reg_data[lags_cols])
y = reg_data["log_gdp"]

model = sm.OLS(y, X).fit()

# Estimation de l'output gap qu'on considere comme etre les residus sous la th d'hamilton
df_hamilton.loc[reg_data.index, "output_gap_hamilton"] = model.resid


In [18]:
fig_outputgap = go.Figure([
        go.Scatter(x = data_detrended['year'],
                   y = data_detrended["output_gap_hamilton"],
                   line = dict(color = "blue"),
                   name = "trend"
        )
])

fig_outputgap.show()